# UC1 Retrieval + RRF Query Merge + Cross-Encoder Evaluation

This notebook runs the full retrieval pipeline in `main.py` and evaluates it with the same logic as `2_evaluation_SOTA_NLP_LIR.ipynb` for scientific comparability.

Comparison target:
- BM25-stage rankings, including baseline BM25 and query-wise BM25 + Reciprocal Rank Fusion (RRF)
- Finished pipeline rankings after cross-encoder reranking, using the original query only for the encoder step

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


In [2]:
import importlib
import main

importlib.reload(main)

# Uses recorded query variants from output_with_agents_uc1.csv, so this does not call the LLM agents.
# To regenerate query variants, switch provider to "ollama" or "gemini" once.
# Project root: output_ranking_uc1.xlsx, output_with_agents_uc1.csv
main.retrieval_pipeline(provider="records", use_case="uc1")

Loaded recorded query variants from output_with_agents_uc1.csv
Loaded 1 queries for level 'process'
[process] Processing row 1/1...
Loaded 7 queries for level 'subprocess'
[subprocess] Processing row 1/7...
[subprocess] Processing row 2/7...
[subprocess] Processing row 3/7...
[subprocess] Processing row 4/7...
[subprocess] Processing row 5/7...
[subprocess] Processing row 6/7...
[subprocess] Processing row 7/7...
Loaded 31 queries for level 'task'
[task] Processing row 1/31...
[task] Processing row 2/31...
[task] Processing row 3/31...
[task] Processing row 4/31...
[task] Processing row 5/31...
[task] Processing row 6/31...
[task] Processing row 7/31...
[task] Processing row 8/31...
[task] Processing row 9/31...
[task] Processing row 10/31...
[task] Processing row 11/31...
[task] Processing row 12/31...
[task] Processing row 13/31...
[task] Processing row 14/31...
[task] Processing row 15/31...
[task] Processing row 16/31...
[task] Processing row 17/31...
[task] Processing row 18/31...

In [3]:
def clean_text(text):
    cleaned_text = str(text).replace("or\n\n\n", " ")
    cleaned_text = cleaned_text.replace("or\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n", " ")
    cleaned_text = cleaned_text.replace("\n \n", " ")
    cleaned_text = cleaned_text.replace("\n", " ")
    return cleaned_text


def evaluate_with_original_logic(df_gs, df_alg):
    df_gs = df_gs.copy()
    df_alg = df_alg.copy()

    df_gs["query_cleaned"] = df_gs.apply(lambda row: clean_text(row["query"]), axis=1)
    df_gs["rel_text_cleaned"] = df_gs.apply(lambda row: clean_text(row["rel_text"]), axis=1)
    df_gs = df_gs.drop(["query", "rel_text"], axis=1)
    df_gs = df_gs.rename(columns={"query_cleaned": "query", "rel_text_cleaned": "rel_text"})

    df_alg["query"] = df_alg["query"].apply(clean_text)
    df_alg["rel_text"] = df_alg["rel_text"].apply(clean_text)
    df_alg["rank"] = df_alg.groupby("query")["score"].rank(ascending=False)

    df_gs_enhanced = pd.merge(
        df_gs,
        df_alg,
        how="left",
        left_on=["query", "rel_text"],
        right_on=["query", "rel_text"],
    )

    df_gs_enhanced["AP"] = 1 / df_gs_enhanced["rank"]
    df_gs_enhanced = df_gs_enhanced.fillna(0)

    map_value = df_gs_enhanced["AP"].mean()
    tp_per_query = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x > 0).sum()).reset_index(name="count")
    fn_per_query = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x == 0).sum()).reset_index(name="count")

    return {
        "map": map_value,
        "avg_tp": tp_per_query["count"].mean(),
        "avg_fn": fn_per_query["count"].mean(),
        "details": df_gs_enhanced,
    }

In [4]:
USE_CASE = "uc1"

new_output_path = project_root / f"output_ranking_{USE_CASE}.xlsx"

gs_paths = {
    "process": project_root
    / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/{USE_CASE}/gold_standard/gs_{USE_CASE}_process_level.xlsx",
    "subprocess": project_root
    / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/{USE_CASE}/gold_standard/gs_{USE_CASE}_subprocess_level.xlsx",
    "task": project_root
    / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/{USE_CASE}/gold_standard/gs_{USE_CASE}_event_level.xlsx",
}

def read_output_sheet(sheet_name):
    try:
        return pd.read_excel(new_output_path, sheet_name=sheet_name)
    except ValueError:
        return pd.DataFrame(columns=["level", "query", "rel_text", "score"])


run_sheets = {
    "bm25_baseline": ("bm25", "BM25_baseline"),
    "bm25_rrf_query_merge": ("bm25", "BM25_RRF_query_merge"),
    "bm25_rrf_weighted": ("bm25", "BM25_RRF_weighted"),
    "bm25_ce": ("finished_pipeline", "BM25_CE"),
    "bm25_rrf_ce": ("finished_pipeline", "BM25_RRF_CE"),
    "bm25_rrf_weighted_ce": ("finished_pipeline", "BM25_RRF_weighted_CE"),
}

run_frames = {
    run_name: {"stage": stage, "df": read_output_sheet(sheet_name)}
    for run_name, (stage, sheet_name) in run_sheets.items()
}


def add_level_if_missing(df):
    if "level" in df.columns:
        return df

    df = df.copy()
    process_queries = set(pd.read_excel(gs_paths["process"])["query"].astype(str).apply(clean_text).tolist())
    subprocess_queries = set(
        pd.read_excel(
            project_root
            / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/input_ranking/{USE_CASE}/Input_queries_medium_{USE_CASE}.xlsx"
        )["process_text"]
        .astype(str)
        .apply(clean_text)
        .tolist()
    )
    task_queries = set(
        pd.read_excel(
            project_root
            / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/input_ranking/{USE_CASE}/Input_queries_low_{USE_CASE}.xlsx"
        )["process_text"]
        .astype(str)
        .apply(clean_text)
        .tolist()
    )

    def infer_level(query):
        q = clean_text(query)
        if q in task_queries:
            return "task"
        if q in subprocess_queries:
            return "subprocess"
        if q in process_queries:
            return "process"
        return "unknown"

    df["level"] = df["query"].apply(infer_level)
    return df[df["level"] != "unknown"].copy()


for run_name, run_data in run_frames.items():
    run_data["df"] = add_level_if_missing(run_data["df"])

all_details = {}
rows = []
for level in ["process", "subprocess", "task"]:
    df_gs = pd.read_excel(gs_paths[level])

    for run_name, run_data in run_frames.items():
        df_run = run_data["df"]
        if df_run.empty:
            continue

        df_level = df_run[df_run["level"] == level].copy()
        run_eval = evaluate_with_original_logic(df_gs, df_level)
        all_details[(level, run_name)] = run_eval["details"]
        rows.append(
            {
                "stage": run_data["stage"],
                "level": level,
                "run": run_name,
                "MAP": run_eval["map"],
                "avg_true_positives": run_eval["avg_tp"],
                "avg_false_negatives": run_eval["avg_fn"],
            }
        )

comparison = pd.DataFrame(rows)
bm25_results = comparison[comparison["stage"] == "bm25"].reset_index(drop=True)
finished_pipeline_results = comparison[comparison["stage"] == "finished_pipeline"].reset_index(drop=True)

print("BM25-stage results")
display(bm25_results)
print("Finished pipeline results")
display(finished_pipeline_results)
comparison

BM25-stage results


,stage,level,run,MAP,avg_true_positives,avg_false_negatives
0,bm25,process,bm25_baseline,0.046035,14.000000,35.000000
1,bm25,process,bm25_rrf_query_merge,0.018974,20.000000,30.000000
2,bm25,process,bm25_rrf_weighted,0.051540,21.000000,29.000000
3,bm25,subprocess,bm25_baseline,0.036449,1.428571,9.142857
4,bm25,subprocess,bm25_rrf_query_merge,0.037294,2.428571,8.142857
5,bm25,subprocess,bm25_rrf_weighted,0.033443,2.000000,8.571429
6,bm25,task,bm25_baseline,0.058357,0.793103,3.793103
7,bm25,task,bm25_rrf_query_merge,0.069524,0.793103,3.793103
8,bm25,task,bm25_rrf_weighted,0.065198,0.793103,3.793103


Finished pipeline results


,stage,level,run,MAP,avg_true_positives,avg_false_negatives
0,finished_pipeline,process,bm25_ce,0.022025,27.000000,22.000000
1,finished_pipeline,process,bm25_rrf_ce,0.030001,20.000000,30.000000
2,finished_pipeline,process,bm25_rrf_weighted_ce,0.025567,21.000000,29.000000
3,finished_pipeline,subprocess,bm25_ce,0.057928,3.428571,7.142857
4,finished_pipeline,subprocess,bm25_rrf_ce,0.059455,2.428571,8.142857
5,finished_pipeline,subprocess,bm25_rrf_weighted_ce,0.050814,2.000000,8.571429
6,finished_pipeline,task,bm25_ce,0.112767,1.068966,3.517241
7,finished_pipeline,task,bm25_rrf_ce,0.090741,0.793103,3.793103
8,finished_pipeline,task,bm25_rrf_weighted_ce,0.101889,0.793103,3.793103


,stage,level,run,MAP,avg_true_positives,avg_false_negatives
0,bm25,process,bm25_baseline,0.046035,14.000000,35.000000
1,bm25,process,bm25_rrf_query_merge,0.018974,20.000000,30.000000
2,bm25,process,bm25_rrf_weighted,0.051540,21.000000,29.000000
3,finished_pipeline,process,bm25_ce,0.022025,27.000000,22.000000
4,finished_pipeline,process,bm25_rrf_ce,0.030001,20.000000,30.000000
5,finished_pipeline,process,bm25_rrf_weighted_ce,0.025567,21.000000,29.000000
6,bm25,subprocess,bm25_baseline,0.036449,1.428571,9.142857
7,bm25,subprocess,bm25_rrf_query_merge,0.037294,2.428571,8.142857
8,bm25,subprocess,bm25_rrf_weighted,0.033443,2.000000,8.571429
9,finished_pipeline,subprocess,bm25_ce,0.057928,3.428571,7.142857


In [5]:
output_dir = (
    project_root
    / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/{USE_CASE}"
)
level_file_names = {
    "process": "process",
    "subprocess": "subprocess",
    "task": "task",
}

for (level, run_name), details in all_details.items():
    output_path = output_dir / f"results_{run_name}_{level_file_names[level]}_{USE_CASE}.xlsx"
    details.to_excel(output_path, index=False)

comparison

,stage,level,run,MAP,avg_true_positives,avg_false_negatives
0,bm25,process,bm25_baseline,0.046035,14.000000,35.000000
1,bm25,process,bm25_rrf_query_merge,0.018974,20.000000,30.000000
2,bm25,process,bm25_rrf_weighted,0.051540,21.000000,29.000000
3,finished_pipeline,process,bm25_ce,0.022025,27.000000,22.000000
4,finished_pipeline,process,bm25_rrf_ce,0.030001,20.000000,30.000000
5,finished_pipeline,process,bm25_rrf_weighted_ce,0.025567,21.000000,29.000000
6,bm25,subprocess,bm25_baseline,0.036449,1.428571,9.142857
7,bm25,subprocess,bm25_rrf_query_merge,0.037294,2.428571,8.142857
8,bm25,subprocess,bm25_rrf_weighted,0.033443,2.000000,8.571429
9,finished_pipeline,subprocess,bm25_ce,0.057928,3.428571,7.142857


In [6]:
PAPER_METHODS = {
    "BM25+CE + weighted query diversification": "BM25_RRF_weighted_CE",
}

LEVEL_LABELS = {
    "process": "level 1: process relevance",
    "subprocess": "level 2: sub-process relevance",
    "task": "level 3: task/event relevance",
}

LEVEL_GS_FILES = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}

USE_CASE_LABELS = {
    "uc1": "use case 1",
    "uc2": "use case 2",
}


def load_paper_inputs(use_case, sheet_name):
    ranking_path = project_root / f"output_ranking_{use_case}.xlsx"
    corpus_path = (
        project_root
        / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/input_ranking/{use_case}/Input_corpus_{use_case}.xlsx"
    )

    if not ranking_path.exists():
        return None, None

    try:
        df_predictions = pd.read_excel(ranking_path, sheet_name=sheet_name)
    except ValueError:
        return None, None

    df_corpus = pd.read_excel(corpus_path)
    corpus_texts = set(df_corpus["requirement_text"].astype(str).apply(clean_text).tolist())
    return df_predictions, corpus_texts


def relevance_metrics_for_level(use_case, level, sheet_name):
    df_predictions, corpus_texts = load_paper_inputs(use_case, sheet_name)
    if df_predictions is None:
        return {"Acc.": np.nan, "Prec.": np.nan, "Rec.": np.nan}

    gs_path = (
        project_root
        / f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{LEVEL_GS_FILES[level]}.xlsx"
    )
    df_gold = pd.read_excel(gs_path)

    df_gold = df_gold.copy()
    df_predictions = df_predictions.copy()
    df_gold["query"] = df_gold["query"].apply(clean_text)
    df_gold["rel_text"] = df_gold["rel_text"].apply(clean_text)
    df_predictions["query"] = df_predictions["query"].apply(clean_text)
    df_predictions["rel_text"] = df_predictions["rel_text"].apply(clean_text)

    if "level" in df_predictions.columns:
        df_predictions = df_predictions[df_predictions["level"] == level].copy()

    queries = sorted(set(df_gold["query"].tolist()) | set(df_predictions["query"].tolist()))
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0

    for query in queries:
        gold_relevant = set(df_gold[df_gold["query"] == query]["rel_text"].tolist())
        predicted_relevant = set(df_predictions[df_predictions["query"] == query]["rel_text"].tolist())

        query_tp = len(gold_relevant & predicted_relevant)
        query_fp = len(predicted_relevant - gold_relevant)
        query_fn = len(gold_relevant - predicted_relevant)
        query_tn = len(corpus_texts - gold_relevant - predicted_relevant)

        true_positives += query_tp
        false_positives += query_fp
        false_negatives += query_fn
        true_negatives += query_tn

    accuracy_denominator = true_positives + false_positives + false_negatives + true_negatives
    precision_denominator = true_positives + false_positives
    recall_denominator = true_positives + false_negatives

    accuracy = (true_positives + true_negatives) / accuracy_denominator if accuracy_denominator else np.nan
    precision = true_positives / precision_denominator if precision_denominator else np.nan
    recall = true_positives / recall_denominator if recall_denominator else np.nan

    return {"Acc.": accuracy, "Prec.": precision, "Rec.": recall}


paper_rows = []
for level in ["process", "subprocess", "task"]:
    for method_name, sheet_name in PAPER_METHODS.items():
        row = {
            ("", "process level"): LEVEL_LABELS[level],
            ("", "method"): method_name,
        }
        for use_case, use_case_label in USE_CASE_LABELS.items():
            metrics = relevance_metrics_for_level(use_case, level, sheet_name)
            for metric_name, metric_value in metrics.items():
                row[(use_case_label, metric_name)] = metric_value
        paper_rows.append(row)

paper_table = pd.DataFrame(paper_rows)
paper_table.columns = pd.MultiIndex.from_tuples(paper_table.columns)
metric_columns = [
    (use_case_label, metric_name)
    for use_case_label in USE_CASE_LABELS.values()
    for metric_name in ["Acc.", "Prec.", "Rec."]
]
paper_table[metric_columns] = paper_table[metric_columns].round(2)

paper_table

\
                    process level                                    method   
0      level 1: process relevance  BM25+CE + weighted query diversification   
1  level 2: sub-process relevance  BM25+CE + weighted query diversification   
2   level 3: task/event relevance  BM25+CE + weighted query diversification   

  use case 1             use case 2              
        Acc. Prec.  Rec.       Acc. Prec.  Rec.  
0       0.79  0.20  0.41       0.73  0.17  0.55  
1       0.93  0.07  0.19       0.89  0.07  0.25  
2       0.96  0.05  0.17       0.94  0.06  0.16